In [1]:
# GUIA 6 PROYECTO - ArcGIS

from pathlib import Path
import ast
import numpy as np
import pandas as pd

BASE = Path('/content/GDIABD')

carpetas = [
    "datos/procesados",
    "datos/geo",
    "documentos/geo",
    "evidencias/geo",
    "visualizaciones",
    "notebooks",
]

for c in carpetas:
    (BASE / c).mkdir(parents=True, exist_ok=True)

(BASE / "README.md").touch()

print("Estructura creada: -----------------")
for p in sorted(BASE.rglob("*")):
    nivel = len(p.relative_to(BASE).parts) - 1
    print(" " * nivel + p.name + ("/" if p.is_dir() else ""))

Estructura creada: -----------------
README.md
datos/
 geo/
 procesados/
documentos/
 geo/
evidencias/
 geo/
notebooks/
visualizaciones/


In [2]:
# SUBIR DATASET LIMPIO
from google.colab import files

print("Sube dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv")
print("Presiona 'Choose Files' y selecciona el archivo\n")

uploaded = files.upload()

nombre = list(uploaded.keys())[0]
ruta_limpio = BASE / "datos" / "procesados" / nombre
ruta_limpio.write_bytes(uploaded[nombre])
print(f"  {nombre} → datos/procesados/")

# LEER DATASET LIMPIO
df_limpio = pd.read_csv(ruta_limpio)

print(f"Dataset limpio: {df_limpio.shape[0]} filas, {df_limpio.shape[1]} columnas")
print(df_limpio["riesgo_academico"].value_counts())

Sube dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv
Presiona 'Choose Files' y selecciona el archivo



Saving dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv to dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv
  dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv → datos/procesados/
Dataset limpio: 1000 filas, 43 columnas
riesgo_academico
Alto     467
Medio    322
Bajo     211
Name: count, dtype: int64


In [3]:
# COORDENADAS APROXIMADAS POR ZONA
COORDENADAS = {
    "San Salvador Central": (13.6929, -89.2182),
    "San Salvador Norte":   (13.8069, -89.1797),
    "San Salvador Sur":     (13.6544, -89.1828),
    "San Salvador Este":    (13.7100, -89.1394),
    "San Salvador Oeste":   (13.7403, -89.2133),
    "La Libertad Centro":   (13.6769, -89.2797),
    "La Libertad Este":     (13.6730, -89.2480),
    "La Libertad Sur":      (13.5867, -89.2872),
    "Sonsonate Centro":     (13.7189, -89.7242),
    "Santa Ana Centro":     (13.9942, -89.5597),
    "San Miguel Centro":    (13.4833, -88.1833),
    "Usulután Este":        (13.3500, -88.4500),
}

# REVISAR QUE TODAS LAS ZONAS TENGAN COORDENADAS
sin_coordenadas = set(df_limpio["municipio_residencia"]) - set(COORDENADAS)
if sin_coordenadas:
    raise ValueError(f"Zonas sin coordenadas: {sorted(sin_coordenadas)}")

print(f"Zonas con coordenadas: {len(COORDENADAS)}")

Zonas con coordenadas: 12


In [4]:
# CREAR DATASET DE MUNICIPIOS

# PORCENTAJE DE UN VALOR EN UNA COLUMNA
def pct(serie, valor):
    return (serie == valor).mean() * 100

# AGRUPAR POR ZONA
agregado = (
    df_limpio.groupby("municipio_residencia")
    .agg(
        zona=("zona", "first"),
        estudiantes=("carnet", "count"),
        promedio_academico=("promedio_academico", "mean"),
        nota_final_prom=("nota_final", "mean"),
        asistencia_prom=("porcentaje_asistencia", "mean"),
        actividad_lms_prom=("actividad_lms", "mean"),
        prob_desercion_prom=("probabilidad_desercion", "mean"),
        horas_trabajo_prom=("horas_trabajo_semana", "mean"),
        pct_riesgo_academico_alto=("riesgo_academico", lambda s: pct(s, "Alto")),
        pct_riesgo_academico_medio=("riesgo_academico", lambda s: pct(s, "Medio")),
        pct_riesgo_academico_bajo=("riesgo_academico", lambda s: pct(s, "Bajo")),
        pct_materia_reprobada=("materia_reprobada", lambda s: s.mean() * 100),
        pct_requiere_seguimiento=("requiere_seguimiento", lambda s: s.mean() * 100),
        pct_trabaja=("trabaja", lambda s: pct(s, "Sí")),
        pct_internet_si=("acceso_internet_casa", lambda s: pct(s, "Sí")),
        pct_internet_limitado=("acceso_internet_casa", lambda s: pct(s, "Limitado (Datos móviles)")),
        pct_internet_no=("acceso_internet_casa", lambda s: pct(s, "No")),
        pct_con_beca=("beca", lambda s: (s != "Sin Beca").mean() * 100),
    )
    .reset_index()
    .rename(columns={"municipio_residencia": "municipio"})
)

# AGREGAR COORDENADAS
agregado["latitud"] = agregado["municipio"].map(lambda m: COORDENADAS[m][0])
agregado["longitud"] = agregado["municipio"].map(lambda m: COORDENADAS[m][1])

# ORDENAR COLUMNAS
primeras = ["municipio", "zona", "latitud", "longitud", "estudiantes"]
columnas_municipios = primeras + [c for c in agregado.columns if c not in primeras]

# REDONDEAR
redondeo = {
    "promedio_academico": 2, "nota_final_prom": 2, "asistencia_prom": 1, "actividad_lms_prom": 1,
    "prob_desercion_prom": 3, "horas_trabajo_prom": 1,
}
redondeo.update({c: 1 for c in agregado.columns if c.startswith("pct_")})

df_municipios = agregado[columnas_municipios].round(redondeo)

print(f"Estudiantes en municipios: {df_municipios['estudiantes'].sum()} (esperado {len(df_limpio)})")
print(df_municipios[["municipio", "zona", "estudiantes", "pct_riesgo_academico_alto"]].to_string(index=False))

Estudiantes en municipios: 1000 (esperado 1000)
           municipio   zona  estudiantes  pct_riesgo_academico_alto
  La Libertad Centro  Rural           81                       35.8
    La Libertad Este Urbana           75                       46.7
     La Libertad Sur Urbana           81                       34.6
   San Miguel Centro Urbana           85                       56.5
San Salvador Central Urbana           75                       45.3
   San Salvador Este Urbana           74                       43.2
  San Salvador Norte Urbana          104                       51.0
  San Salvador Oeste Urbana           72                       44.4
    San Salvador Sur Urbana           83                       53.0
    Santa Ana Centro Urbana           84                       44.0
    Sonsonate Centro Urbana           95                       49.5
       Usulután Este  Rural           91                       52.7


In [5]:
# CREAR DATASET DE ALERTAS

# FECHA DE LA ULTIMA INTERVENCION
def ultima_fecha(json_str):
    try:
        data = ast.literal_eval(json_str) if isinstance(json_str, str) else json_str
        fechas = pd.to_datetime([e.get("fecha") for e in data.get("bitacora", [])], errors="coerce").dropna()
        return fechas.max() if len(fechas) else pd.NaT
    except Exception:
        return pd.NaT

#  DEFINIR COLUMNAS
columnas_alertas = [
    "facultad", "carrera", "turno", "ciclo_actual", "materia",
    "riesgo_academico", "probabilidad_desercion",
    "nota_teoria", "nota_laboratorio", "nota_final", "promedio_academico", "materia_reprobada",
    "porcentaje_asistencia", "faltas_consecutivas",
    "tareas_entregadas", "tareas_tarde", "participacion_nivel", "actividad_lms",
    "acceso_internet_casa", "trabaja", "horas_trabajo_semana", "beca",
    "observacion_docente", "total_observaciones", "requiere_seguimiento",
    "ultima_intervencion", "estado_ultimo",
]

df_alertas_gen = df_limpio[columnas_alertas].copy()
df_alertas_gen.insert(0, "municipio_referencia", df_limpio["municipio_residencia"])
df_alertas_gen.insert(1, "zona", df_limpio["zona"])
df_alertas_gen["fecha"] = df_limpio["historial_cualitativo"].apply(ultima_fecha).dt.strftime("%Y-%m-%d")

# DISPERSAR LOS PUNTOS ALREDEDOR DE SU ZONA (SEMILLA FIJA)
rng = np.random.default_rng(42)
dispersion_lat = np.clip(rng.normal(0, 0.012, len(df_alertas_gen)), -0.03, 0.03)
dispersion_lon = np.clip(rng.normal(0, 0.012, len(df_alertas_gen)), -0.03, 0.03)

base_lat = df_alertas_gen["municipio_referencia"].map(lambda m: COORDENADAS[m][0])
base_lon = df_alertas_gen["municipio_referencia"].map(lambda m: COORDENADAS[m][1])

df_alertas_gen.insert(2, "latitud", (base_lat + dispersion_lat).round(5))
df_alertas_gen.insert(3, "longitud", (base_lon + dispersion_lon).round(5))
df_alertas_gen.insert(0, "id_evento", range(1, len(df_alertas_gen) + 1))

print(f"Alertas: {len(df_alertas_gen)} filas, {df_alertas_gen.shape[1]} columnas")
print(f"Fechas vacias (sin intervencion): {df_alertas_gen['fecha'].isna().sum()}")

Alertas: 1000 filas, 33 columnas
Fechas vacias (sin intervencion): 157


In [6]:
# GUARDAR CSV
ruta = BASE / "datos" / "geo" / "municipios_alertacampus_guia06.csv"
ruta_alertas = BASE / "datos" / "geo" / "alertas_sinteticas_alertacampus_guia06.csv"

df_municipios.to_csv(ruta, index=False, encoding="utf-8")
df_alertas_gen.to_csv(ruta_alertas, index=False, encoding="utf-8")

print(f"Guardado: {ruta.name} ({len(df_municipios)} filas)")
print(f"Guardado: {ruta_alertas.name} ({len(df_alertas_gen)} filas)")

Guardado: municipios_alertacampus_guia06.csv (12 filas)
Guardado: alertas_sinteticas_alertacampus_guia06.csv (1000 filas)


In [7]:
# LEER CSV DE MUNICIPIOS
df = pd.read_csv(ruta)

# LEER CSV DE ALERTAS
df_alertas = pd.read_csv(ruta_alertas)

# IMPRIMIR INFORMACION DE MUNICIPIOS
print("\n------Archivo Municipios ------\n")
print(f"Registros: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print(df.head())

# IMPRIMIR INFORMACION DE ALERTAS
print("\n------Archivo Alertas ------\n")
print(f"Registros: {len(df_alertas)}")
print(f"Columnas: {df_alertas.columns.tolist()}")
print(df_alertas.head())


------Archivo Municipios ------

Registros: 12
Columnas: ['municipio', 'zona', 'latitud', 'longitud', 'estudiantes', 'promedio_academico', 'nota_final_prom', 'asistencia_prom', 'actividad_lms_prom', 'prob_desercion_prom', 'horas_trabajo_prom', 'pct_riesgo_academico_alto', 'pct_riesgo_academico_medio', 'pct_riesgo_academico_bajo', 'pct_materia_reprobada', 'pct_requiere_seguimiento', 'pct_trabaja', 'pct_internet_si', 'pct_internet_limitado', 'pct_internet_no', 'pct_con_beca']
              municipio    zona  latitud  longitud  estudiantes  \
0    La Libertad Centro   Rural  13.6769  -89.2797           81   
1      La Libertad Este  Urbana  13.6730  -89.2480           75   
2       La Libertad Sur  Urbana  13.5867  -89.2872           81   
3     San Miguel Centro  Urbana  13.4833  -88.1833           85   
4  San Salvador Central  Urbana  13.6929  -89.2182           75   

   promedio_academico  nota_final_prom  asistencia_prom  actividad_lms_prom  \
0                7.11             7.11

In [8]:
# CONVERTIR COORDENADAS A NUMERO
df["latitud"] = pd.to_numeric(df["latitud"], errors="coerce")
df["longitud"] = pd.to_numeric(df["longitud"], errors="coerce")

df_alertas["latitud"] = pd.to_numeric(df_alertas["latitud"], errors="coerce")
df_alertas["longitud"] = pd.to_numeric(df_alertas["longitud"], errors="coerce")

print("Coordenadas convertidas a numerico --------------------")
print("-----Municipios -----")
print(df[["municipio", "latitud", "longitud"]])

Coordenadas convertidas a numerico --------------------
-----Municipios -----
               municipio  latitud  longitud
0     La Libertad Centro  13.6769  -89.2797
1       La Libertad Este  13.6730  -89.2480
2        La Libertad Sur  13.5867  -89.2872
3      San Miguel Centro  13.4833  -88.1833
4   San Salvador Central  13.6929  -89.2182
5      San Salvador Este  13.7100  -89.1394
6     San Salvador Norte  13.8069  -89.1797
7     San Salvador Oeste  13.7403  -89.2133
8       San Salvador Sur  13.6544  -89.1828
9       Santa Ana Centro  13.9942  -89.5597
10      Sonsonate Centro  13.7189  -89.7242
11         Usulután Este  13.3500  -88.4500


In [9]:
# VALIDAR QUE LAS COORDENADAS ESTEN EN EL SALVADOR
def validar_coordenadas(datos, nombre):
    mascara = (
        datos["latitud"].between(13, 15) &
        datos["longitud"].between(-91, -87)
    )

    print(f"----- {nombre} -----")
    print(f"Total de registros: {len(datos)}")
    print(f"Coordenadas validas: {mascara.sum()}")
    print(f"Coordenadas a revisar: {(~mascara).sum()}")

    if (~mascara).sum() > 0:
        print("\nCoordenadas Invalidas: ----------")
        print(datos[~mascara])
    else:
        print("Todas las coordenadas estan dentro de El Salvador ----------")
    print()
    return mascara


mask_valida = validar_coordenadas(df, "Municipios")
mask_valida_alertas = validar_coordenadas(df_alertas, "Alertas")

----- Municipios -----
Total de registros: 12
Coordenadas validas: 12
Coordenadas a revisar: 0
Todas las coordenadas estan dentro de El Salvador ----------

----- Alertas -----
Total de registros: 1000
Coordenadas validas: 1000
Coordenadas a revisar: 0
Todas las coordenadas estan dentro de El Salvador ----------



In [10]:
# REVISAR NULOS EN MUNICIPIOS
print("Nulos en municipios ---------")
print(df.isna().sum())

# REVISAR DUPLICADOS EN MUNICIPIOS
print(f"\nDuplicados en municipios: {df.duplicated().sum()}")

# REVISAR COORDENADAS REPETIDAS EN MUNICIPIOS
duplicados_coords = df.duplicated(subset=["latitud", "longitud"]).sum()
print(f"Coordenadas repetidas: {duplicados_coords}")

# REVISAR NULOS EN ALERTAS
print("\nNulos en alertas ----")
print(df_alertas.isna().sum())

# REVISAR DUPLICADOS EN ALERTAS
print(f"\nDuplicados en alertas: {df_alertas.duplicated().sum()}")
print(f"Identificadores repetidos: {df_alertas['id_evento'].duplicated().sum()}")

Nulos en municipios ---------
municipio                     0
zona                          0
latitud                       0
longitud                      0
estudiantes                   0
promedio_academico            0
nota_final_prom               0
asistencia_prom               0
actividad_lms_prom            0
prob_desercion_prom           0
horas_trabajo_prom            0
pct_riesgo_academico_alto     0
pct_riesgo_academico_medio    0
pct_riesgo_academico_bajo     0
pct_materia_reprobada         0
pct_requiere_seguimiento      0
pct_trabaja                   0
pct_internet_si               0
pct_internet_limitado         0
pct_internet_no               0
pct_con_beca                  0
dtype: int64

Duplicados en municipios: 0
Coordenadas repetidas: 0

Nulos en alertas ----
id_evento                   0
municipio_referencia        0
zona                        0
latitud                     0
longitud                    0
facultad                    0
carrera                     

In [11]:
# REVISAR QUE LOS MUNICIPIOS DE LAS ALERTAS EXISTAN EN MUNICIPIOS
sin_municipio = sorted(set(df_alertas["municipio_referencia"]) - set(df["municipio"]))
print(f"Municipios de alertas sin correspondencia: {sin_municipio if sin_municipio else 'ninguno'}")

# CONVERTIR FECHA A FORMATO FECHA
df_alertas["fecha"] = pd.to_datetime(df_alertas["fecha"], format="%Y-%m-%d", errors="coerce")
print(f"Fechas vacias: {df_alertas['fecha'].isna().sum()}")
print(f"Rango de fechas: {df_alertas['fecha'].min().date()} a {df_alertas['fecha'].max().date()}")

# REVISAR CATEGORIAS PARA FILTROS Y SIMBOLOGIA
print("\nZona (municipios):")
print(df["zona"].value_counts())
print("\nRiesgo academico (alertas):")
print(df_alertas["riesgo_academico"].value_counts())
print("\nUltima intervencion (alertas):")
print(df_alertas["ultima_intervencion"].value_counts())

Municipios de alertas sin correspondencia: ninguno
Fechas vacias: 157
Rango de fechas: 2026-01-10 a 2026-08-28

Zona (municipios):
zona
Urbana    10
Rural      2
Name: count, dtype: int64

Riesgo academico (alertas):
riesgo_academico
Alto     467
Medio    322
Bajo     211
Name: count, dtype: int64

Ultima intervencion (alertas):
ultima_intervencion
Firma de Compromiso Académico    191
Seguimiento Psicológico          175
Reunión de Orientación           164
Llamada a Encargado/Apostilla    161
Sin intervención                 157
Tutoría Académica                152
Name: count, dtype: int64


In [12]:
print("Tipos de datos en df (municipios):\n")
df.info()

print("\nTipos de datos en df_alertas (alertas):\n")
df_alertas.info()

Tipos de datos en df (municipios):

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   municipio                   12 non-null     object 
 1   zona                        12 non-null     object 
 2   latitud                     12 non-null     float64
 3   longitud                    12 non-null     float64
 4   estudiantes                 12 non-null     int64  
 5   promedio_academico          12 non-null     float64
 6   nota_final_prom             12 non-null     float64
 7   asistencia_prom             12 non-null     float64
 8   actividad_lms_prom          12 non-null     float64
 9   prob_desercion_prom         12 non-null     float64
 10  horas_trabajo_prom          12 non-null     float64
 11  pct_riesgo_academico_alto   12 non-null     float64
 12  pct_riesgo_academico_medio  12 non-null     float64
 13  p

In [13]:
# EXPORTAR VERSIONES VALIDADAS (SOLO COORDENADAS VALIDAS)
df_valido = df[mask_valida].copy()
df_alertas_valido = df_alertas[mask_valida_alertas].copy()
df_alertas_valido["fecha"] = df_alertas_valido["fecha"].dt.strftime("%Y-%m-%d")

ruta_salida = BASE / "datos" / "geo" / "municipios_alertacampus_guia06_validado.csv"
ruta_salida_alertas = BASE / "datos" / "geo" / "alertas_sinteticas_alertacampus_guia06_validado.csv"

df_valido.to_csv(ruta_salida, index=False, encoding="utf-8")
df_alertas_valido.to_csv(ruta_salida_alertas, index=False, encoding="utf-8")

print(f"Archivo validado guardado: {ruta_salida}")
print(f"Archivo validado guardado: {ruta_salida_alertas}")
print(f"Registros excluidos: municipios {len(df) - len(df_valido)}, alertas {len(df_alertas) - len(df_alertas_valido)}")

# DESCARGAR
from google.colab import files

files.download(str(ruta_salida))
files.download(str(ruta_salida_alertas))

Archivo validado guardado: /content/GDIABD/datos/geo/municipios_alertacampus_guia06_validado.csv
Archivo validado guardado: /content/GDIABD/datos/geo/alertas_sinteticas_alertacampus_guia06_validado.csv
Registros excluidos: municipios 0, alertas 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>